In [1]:
import numba as nb
from numba import cuda
import math
import numpy as np

# Local (RTX4080)

In [2]:
blocks_per_grid = 400
threads_per_block = 512

@cuda.jit
def partial_sum_fn(arr, partial_sum):
    worker_idx = cuda.grid(1)
    stride = cuda.gridsize(1)

    thread_sum = 0
    for i in range(worker_idx, arr.size, stride):
        thread_sum += arr[i]
    
    block_thread_val = cuda.threadIdx.x
    block_sums = cuda.shared.array(shape = threads_per_block, dtype = nb.float32)
    block_sums[block_thread_val] = thread_sum

    cuda.syncthreads()

    block_thread_val = cuda.threadIdx.x
    size = cuda.blockDim.x // 2
    while size != 0:
        if block_thread_val < size:
            block_sums[block_thread_val] += block_sums[block_thread_val + size]
        cuda.syncthreads()
        size = size // 2

    if block_thread_val == 0:
        partial_sum[cuda.blockIdx.x] = block_sums[block_thread_val]

@cuda.jit
def final_sum_fn(partial_sum, total):
    worker_idx = cuda.grid(1)
    stride = cuda.gridsize(1)

    for i in range(worker_idx, partial_sum.size, stride):
        total[0] += partial_sum[i]

In [3]:
arr_host = np.random.standard_normal(size = (100_000_000)).astype(np.float32)

arr_host.sum()

np.float32(-4141.6113)

In [5]:
arr_dev = cuda.to_device(arr_host)
partial_sum = cuda.device_array(shape = blocks_per_grid, dtype = np.float32)
total = cuda.device_array(shape = 1, dtype = np.float32)

partial_sum_fn[blocks_per_grid, threads_per_block](arr_dev, partial_sum)
final_sum_fn[1, 1](partial_sum, total)

/home/alvinng44/.local/share/mamba/envs/alyssa/lib/python3.12/site-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [6]:
total_cpu = total.copy_to_host()
total_cpu

array([-4141.6123], dtype=float32)

# NSCC (A100)

In [3]:
arr_host = np.random.standard_normal(size = (100_000_000)).astype(np.float32)

arr_host.sum()

np.float32(-10236.678)

In [4]:
arr_dev = cuda.to_device(arr_host)
partial_sum = cuda.device_array(shape = blocks_per_grid, dtype = np.float32)
total = cuda.device_array(shape = 1, dtype = np.float32)

partial_sum_fn[blocks_per_grid, threads_per_block](arr_dev, partial_sum)
final_sum_fn[1, 1](partial_sum, total)

/home/users/nus/e0958343/.conda/envs/alyssa/lib/python3.12/site-packages/numba_cuda/numba/cuda/dispatcher.py:690: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [5]:
total_cpu = total.copy_to_host()
total_cpu

array([-10236.683], dtype=float32)

In [6]:
cuda.is_available()

True

In [7]:
cuda.detect()

Found 1 CUDA devices
id 0    NVIDIA A100-SXM4-40GB                              [SUPPORTED]
                      Compute Capability: 8.0
                           PCI Device ID: 0
                              PCI Bus ID: 65
                                    UUID: GPU-d62df84e-f017-3a07-5492-ec26b529f468
                                Watchdog: Disabled
             FP32/FP64 Performance Ratio: 2
Summary:
	1/1 devices are supported


True

# DSO 

In [3]:
arr_host = np.random.standard_normal(size = (100_000_000)).astype(np.float32)

arr_host.sum()

np.float32(3413.1238)

In [4]:
arr_dev = cuda.to_device(arr_host)
partial_sum = cuda.device_array(shape = blocks_per_grid, dtype = np.float32)
total = cuda.device_array(shape = 1, dtype = np.float32)

partial_sum_fn[blocks_per_grid, threads_per_block](arr_dev, partial_sum)
final_sum_fn[1, 1](partial_sum, total)

/opt/py_venv/lib/python3.12/site-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [5]:
total_cpu = total.copy_to_host()
total_cpu

array([3413.124], dtype=float32)

In [6]:
cuda.is_available()

True

In [7]:
cuda.detect()

Found 3 CUDA devices
id 0    b'NVIDIA RTX 6000 Ada Generation'                              [SUPPORTED]
                      Compute Capability: 8.9
                           PCI Device ID: 0
                              PCI Bus ID: 22
                                    UUID: GPU-6b254f0d-1053-63ce-58f4-72f6a129ee59
                                Watchdog: Enabled
             FP32/FP64 Performance Ratio: 64
id 1    b'NVIDIA RTX 6000 Ada Generation'                              [SUPPORTED]
                      Compute Capability: 8.9
                           PCI Device ID: 0
                              PCI Bus ID: 112
                                    UUID: GPU-5c01064d-5097-b621-22a3-801b56148df0
                                Watchdog: Enabled
             FP32/FP64 Performance Ratio: 64
id 2    b'NVIDIA RTX 6000 Ada Generation'                              [SUPPORTED]
                      Compute Capability: 8.9
                           PCI Device ID: 0
             

True